In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

In [2]:

# --- 1. Load and Prepare Data ---
data = pd.read_csv("OND.csv")

In [3]:
# Features (all except last 3 price columns) and Labels (last 3 columns)
X = data.drop(columns=["Min_Price", "Max_Price", "Modal_Price"])
y = data[["Min_Price", "Max_Price", "Modal_Price"]]

In [4]:
# Convert 'Arrival_Date' to datetime and extract features
X['Arrival_Date'] = pd.to_datetime(X['Arrival_Date'], format='%d-%m-%Y')
X['Day'] = X['Arrival_Date'].dt.day
X['Month'] = X['Arrival_Date'].dt.month
X['Year'] = X['Arrival_Date'].dt.year
X = X.drop(columns=['Arrival_Date'])

In [5]:
# Encode categorical variables
X = X.apply(lambda col:LabelEncoder().fit_transform(col) if col.dtypes == 'object' else col)

In [6]:
# --- 2. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train.head(10)

,State,District,Market,Commodity,Variety,Grade,Day,Month,Year
3767,0,0,13,0,3,0,26,12,2024
379,0,0,19,0,3,0,9,2,2024
6572,0,0,17,0,3,0,7,1,2025
3955,0,0,21,0,1,0,6,3,2024
177,0,0,39,0,1,0,8,8,2024
1643,0,0,11,0,1,1,12,8,2025
132,0,0,13,0,1,0,20,8,2024
4071,0,0,7,0,1,0,22,3,2024
6205,0,0,27,0,1,0,30,8,2024
1374,0,0,25,0,1,0,16,11,2024


In [8]:
# --- 3. Train Random Forest Model (multi-output regression) ---
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=42)

In [9]:
# --- 4. Predictions ---
y_pred = model.predict(X_test)

In [10]:
for i, col in enumerate(y.columns):
# --- 5. Evaluation ---
    print(f"\n=== {col} ===")
    print("R² Score:", r2_score(y_test.iloc[:, i], y_pred[:, i]))
    print("MSE:", mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))


=== Min_Price ===
R² Score: 0.8689868095037835
MSE: 81456.82387986958

=== Max_Price ===
R² Score: 0.9653883219701239
MSE: 53324.147350511674

=== Modal_Price ===
R² Score: 0.9736740423390483
MSE: 33940.40102959261


In [11]:
# Example prediction for the first row of test data
print("\nExample prediction vs actual:")
print("Predicted:", y_pred[1])
print("Actual   :", y_test.iloc[1].values)


Example prediction vs actual:
Predicted: [ 844.18333333 1925.34183333 1465.56875   ]
Actual   : [ 800. 1904. 1500.]


In [12]:
accuracy=model.score(X_test,y_test)
print(f"\nAccuracy: {(accuracy*100):.2f}%")


Accuracy: 93.60%


In [13]:
new_data = pd.DataFrame([
    {
        "Market": "Pune",
        "Commodity": "Onion",
        "Variety": "Red",
        "Grade": "A",
        "Arrival_Date": "03-09-2025",  # Today's date
        "District": "Pune",
        "State": "Maharashtra"
    },
    {
        "Market": "Nashik",
        "Commodity": "Onion",
        "Variety": "White",
        "Grade": "B",
        "Arrival_Date": "04-09-2025",  # Tomorrow's date
        "District": "Nashik",
        "State": "Maharashtra"
    },    {
        "Market": "Mungase",
        "Commodity": "Onion",
        "Variety": "Red",
        "Grade": "FAQ",
        "Arrival_Date": "17-01-2026",  # Tomorrow's date
        "District": "Nashik",
        "State": "Maharashtra"
    },
        {
        "Market": "Devala",
        "Commodity": "Onion",
        "Variety": "other",
        "Grade": "FAQ",
        "Arrival_Date": "26-07-2024",  # Tomorrow's date
        "District": "Nashik",
        "State": "Maharashtra"
    },

])
new_data


,Market,Commodity,Variety,Grade,Arrival_Date,District,State
0,Pune,Onion,Red,A,03-09-2025,Pune,Maharashtra
1,Nashik,Onion,White,B,04-09-2025,Nashik,Maharashtra
2,Mungase,Onion,Red,FAQ,17-01-2026,Nashik,Maharashtra
3,Devala,Onion,other,FAQ,26-07-2024,Nashik,Maharashtra


In [14]:

# Process new data in the same way as training
new_data_processed = new_data.copy()
new_data_processed['Arrival_Date'] = pd.to_datetime(new_data_processed['Arrival_Date'], format='%d-%m-%Y')
new_data_processed['Day'] = new_data_processed['Arrival_Date'].dt.day
new_data_processed['Month'] = new_data_processed['Arrival_Date'].dt.month
new_data_processed['Year'] = new_data_processed['Arrival_Date'].dt.year
new_data_processed = new_data_processed.drop(columns=['Arrival_Date'])

In [21]:


New_Pred = X.apply(lambda col:LabelEncoder().fit_transform(col) if col.dtypes == 'object' else col)

In [23]:

safe_length = min(len(new_pred), len(new_data))

# Optional: Alert you if there is a mismatch
if len(new_pred) != len(new_data):
    print(f"Note: Mismatch detected. Predictions: {len(new_pred)}, Data Rows: {len(new_data)}")
    print(f"Showing first {safe_length} results only.\n")

# 2. Run the loop using the safe range
for i in range(safe_length):
    row = new_pred[i]
    original_info = new_data.iloc[i]
    
    print(f"\n--- Prediction for new sample {i+1} ---")
    print(f"Market: {original_info['Market']}, Variety: {original_info['Variety']}, Date: {original_info['Arrival_Date']}")
    print(f"Predicted Min Price   : ₹{row[0]:.2f}")
    print(f"Predicted Max Price   : ₹{row[1]:.2f}")
    print(f"Predicted Modal Price : ₹{row[2]:.2f}")

Note: Mismatch detected. Predictions: 9393, Data Rows: 4
Showing first 4 results only.


--- Prediction for new sample 1 ---
Market: Pune, Variety: Red, Date: 03-09-2025
Predicted Min Price   : ₹1556.62
Predicted Max Price   : ₹3506.15
Predicted Modal Price : ₹3366.20

--- Prediction for new sample 2 ---
Market: Nashik, Variety: White, Date: 04-09-2025
Predicted Min Price   : ₹1295.26
Predicted Max Price   : ₹2914.93
Predicted Modal Price : ₹2693.40

--- Prediction for new sample 3 ---
Market: Mungase, Variety: Red, Date: 17-01-2026
Predicted Min Price   : ₹1219.57
Predicted Max Price   : ₹2891.85
Predicted Modal Price : ₹2688.18

--- Prediction for new sample 4 ---
Market: Devala, Variety: other, Date: 26-07-2024
Predicted Min Price   : ₹1374.05
Predicted Max Price   : ₹3204.38
Predicted Modal Price : ₹2949.93


In [ ]:
# new_data = pd.DataFrame([
#     {
#         "Market": "Pune",
#         "Commodity": "Onion",
#         "Variety": "Red",
#         "Grade": "A",
#         "Arrival_Date": "03-09-2025",  # Today's date
#         "District": "Pune",
#         "State": "Maharashtra"
#     },
#     {
#         "Market": "Nashik",
#         "Commodity": "Onion",
#         "Variety": "White",
#         "Grade": "B",
#         "Arrival_Date": "04-09-2025",  # Tomorrow's date
#         "District": "Nashik",
#         "State": "Maharashtra"
#     }
# ])

# print("\n--- Predicting Prices for New Data ---")

# # Process new data in the same way as training
# new_data_processed = new_data.copy()
# new_data_processed['Arrival_Date'] = pd.to_datetime(new_data_processed['Arrival_Date'], format='%d-%m-%Y')
# new_data_processed['Day'] = new_data_processed['Arrival_Date'].dt.day
# new_data_processed['Month'] = new_data_processed['Arrival_Date'].dt.month
# new_data_processed['Year'] = new_data_processed['Arrival_Date'].dt.year
# new_data_processed = new_data_processed.drop(columns=['Arrival_Date'])

# # ⚠️ CRITICAL FIX: Use the SAME fitted encoder (from training)
# # cat_cols must be defined during training: e.g.
# cat_cols = ["Market", "Commodity", "Variety", "Grade", "District", "State"]
# # encoder = OrdinalEncoder()
# new_data_processed[cat_cols] = OrdinalEncoder().transform(new_data_processed[cat_cols])

# # Ensure same column order as training
# new_data_processed = new_data_processed[X_train.columns]

# # Predict
# new_pred = model.predict(new_data_processed)

# for i, row in enumerate(new_pred):
#     print(f"\n--- Prediction for new sample {i+1} ---")
#     original_info = new_data.iloc[i]
#     print(f"Market: {original_info['Market']}, Variety: {original_info['Variety']}, Date: {original_info['Arrival_Date']}")
#     print(f"Predicted Min Price  : ₹{row[0]:.2f}")
#     print(f"Predicted Max Price  : ₹{row[1]:.2f}")
#     print(f"Predicted Modal Price: ₹{row[2]:.2f}")


--- Predicting Prices for New Data ---


NotFittedError: This OrdinalEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.